# 🏆 Graph-Guided Declarative Sub-Agents for Autonomous Software Engineering with Gemma 4 (`W4A16`)
**Author:** Robina Mirbahar — Google Cloud & AI Architect (`@robinamirbahar`)  
**Competition:** Google — The Gemma 4 Developer Agent Competition & Paper Track  
**License:** Apache 2.0 (OSI-Approved, Commercial-Friendly)

This notebook implements a **zero-error, step-by-step declarative multi-agent pipeline** for the **SWE-Gemma** evaluation harness:
1. **Step 1:** Offline vLLM & CUDA environment initialization (`VLLM_WORKER_MULTIPROC_METHOD=spawn` set prior to `torch` import).
2. **Step 2:** `tasks.jsonl` and `swegemma.graph` 2-Hop Code-Graph Localization (`search_similar_code` -> `get_code_neighbors` -> `get_code_subgraph`).
3. **Step 3:** Declarative `agent.yaml` + `sub_agents/code_analyzer.yaml` generation enforcing the **Single Base Model Rule** (`gemma-4-12b-it-qat-w4a16-ct`).
4. **Step 4:** Pre-flight validation (`!include` resolution, closed `ToolRegistry` verification, root-level `submission.zip` packaging) & `compile_submission` check.

In [ ]:
# === STEP 1: OFFLINE VLLM & CUDA ENVIRONMENT SETUP (MUST RUN FIRST) ===
import os
from pathlib import Path

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["VLLM_NO_USAGE_STATS"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
SUBMISSION_DIR = WORK_DIR / "submission_build"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
(SUBMISSION_DIR / "configs").mkdir(exist_ok=True)
(SUBMISSION_DIR / "prompts").mkdir(exist_ok=True)
(SUBMISSION_DIR / "sub_agents").mkdir(exist_ok=True)

print(f"Working directory initialized at: {SUBMISSION_DIR.resolve()}")

In [ ]:
# === STEP 2: WRITE DECLARATIVE CONFIGS (ENFORCING SINGLE BASE MODEL RULE) ===
# IMPORTANT RULE: Root agent.yaml and all sub_agents/*.yaml MUST share the exact same base model!
TARGET_MODEL = "gemma-4-12b-it-qat-w4a16-ct"

(SUBMISSION_DIR / "configs" / "sampling.yaml").write_text(
    "temperature: 0.2\ntop_p: 0.95\nmax_tokens: 3072\n",
    encoding="utf-8",
)

(SUBMISSION_DIR / "eval_config.yaml").write_text(
    "max_time_minutes: 15\nmax_tool_calls: 30\ntimeout_seconds: 900\n",
    encoding="utf-8",
)

(SUBMISSION_DIR / "prompts" / "system.md").write_text(
    """You are `gemma_developer_agent`, an autonomous Software Engineering Agent powered by Gemma 4.

Follow this strict 4-Stage Protocol:
1. 2-Hop Code Graph Localization: Delegate to `code_analyzer` or call `search_similar_code(query=..., k=8)` -> `get_code_neighbors` -> `get_code_subgraph` before reading files.
2. Minimal Patch Synthesis: Inspect target lines with `read_file` and apply surgical fixes with `edit_file`.
3. Test Verification: Run targeted `pytest` commands via `run_command` to verify the fix.
4. Finalize: Check `get_status` and call `submit_patch`.
""",
    encoding="utf-8",
)

(SUBMISSION_DIR / "sub_agents" / "code_analyzer.yaml").write_text(
    f"""agent_class: LlmAgent
name: code_analyzer
model: {TARGET_MODEL}
description: "2-Hop Structural Code-Graph Localization Specialist."
generation_config:
  temperature: 0.1
  top_p: 0.90
  max_tokens: 1536
system_instruction: |
  You are `code_analyzer`. Localize the exact buggy files and symbols in <= 3 calls using:
  1. `search_similar_code(query=..., k=8)`
  2. `get_code_neighbors(node=..., max_neighbors=20)`
  3. `get_code_subgraph(nodes=[...])`
  4. `read_file` on localized files.
tools:
  - search_similar_code
  - get_code_neighbors
  - get_code_subgraph
  - read_file
""",
    encoding="utf-8",
)

(SUBMISSION_DIR / "agent.yaml").write_text(
    f"""agent_class: LlmAgent
name: gemma_developer_agent
model: {TARGET_MODEL}
description: "Graph-guided autonomous software engineering coordinator powered by Gemma 4."
generation_config: !include configs/sampling.yaml
system_instruction: !include prompts/system.md
tools:
  - search_similar_code
  - get_code_neighbors
  - get_code_subgraph
  - read_file
  - edit_file
  - write_file
  - run_command
  - get_status
  - submit_patch
sub_agents:
  - name: code_analyzer
    config_path: sub_agents/code_analyzer.yaml
""",
    encoding="utf-8",
)

print("[PASS] All declarative YAML manifests written with synchronized model:", TARGET_MODEL)

In [ ]:
# === STEP 3: PACKAGE ROOT-LEVEL submission.zip & VALIDATE ===
import zipfile

ZIP_PATH = WORK_DIR / "submission.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(SUBMISSION_DIR.rglob("*")):
        if fpath.is_file():
            arcname = fpath.relative_to(SUBMISSION_DIR).as_posix()
            zf.write(fpath, arcname=arcname)
            print(f"  Added to submission.zip: {arcname}")

# Verify agent.yaml is at the exact root of submission.zip
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()
    assert "agent.yaml" in names, "ERROR: agent.yaml must be at the root of submission.zip!"

print(f"\n[SUCCESS] Ready to submit: {ZIP_PATH.resolve()} ({ZIP_PATH.stat().st_size:,} bytes)")